# Idea

Suppose we have a simulator $y = f(x, \theta)$ for experimental settings $x$ and unknown parameters $\theta$ and we want to use it to guide an experiment. We are able to take (potentially expensive) measurements $\mathcal{D} = \{(x, y) \} = \{y_{x}\}$.

We decompose the problem as follows: given measurements $\mathcal{D}$ we want to infer parameters $\theta$ that are consistent with the measurements, namely $p(\theta | \mathcal{D})$. We will do so using simulation-based inference (SBI).

Given this posterior, we then want to decide where to measure next in the real experiment. If the objective is just to sharpen/improve this posterior over parameters $p(\theta | \mathcal{D})$, we can formulate an acquisition function as finding the measurement conditions, $x$, that maximize the expected information gain (EIG) about $\theta$, where EIG is defined as

\begin{equation}
EIG(x) = \mathbb{E}_{\theta\sim p(\theta | \mathcal{D})}\mathbb{E}_{y \sim p(y | x, \theta)}\log\frac{p(\theta | \mathcal{D}, x, y)}{p(\theta | \mathcal{D})}.
\end{equation}

This is a standard Bayesian Optimal Experimental Design (BOED) objective.


For a given candidate setting $x$ we can calculate the EIG via a Monte Carlo estimate as follows:
1) Use SBI to estimate the posterior $p(\theta | \mathcal{D})$ (or start from prior $p(\theta)$ on iter 0).
2) Sample $\theta_{i} \sim p(\theta | \mathcal{D})$.
3) Simulate $y_i \sim p(y | \theta_i, x)$ using the simulator.
4) Use SBI to estimate the posterior $p(\theta | \mathcal{D} \cup \{(x, y_i)\})$.
5) Calculate $\log p(\theta | \mathcal{D} \cup \{(x, y_i)\}) - \log  p(\theta | \mathcal{D})$.
6) Run many times and average.


To find the argmax, this can be done over a grid of $x$, or as part of an optimization procedure with this as the inner loop.


To do a more targeted approach at an experimental objective, we can try to put this into a BAX loop. In this case, we want to not just learn more about $\theta$, but also find settings $x$ that meet some particular (computable) criterion, where such $x$ are represented as $\mathcal{O}_{\mathcal{A}}$. We focus on the case of subsets, namely the output of an algorithm is some $\mathcal{T}$ in the input ($X$ space). Following eq 4 in the InfoBAX paper, we write the EIG about the algorithm output as

\begin{equation}
EIG(x) = \mathbb{H}[y_{x} | \mathcal{D}] - \mathbb{E}_{p(\mathcal{T}|\mathcal{D})}[\mathbb{H}[y_x | \mathcal{D}, \mathcal{T}]]
\end{equation}

For the second term
1) Sample $\theta_i \sim p(\theta | \mathcal{D})$
2) Run algo on $\theta_i \implies \mathcal{T}_i \sim p(\mathcal{T} | \mathcal{D})$
3) Use simulator to evaluate points in $T_{i}$, given $\theta_i$ $(y_j \sim p(y | \theta_i, x_j)$ for $x_j\in \mathcal{T}_i$). $\mathcal{D}_{i}^{hall} = \{(x_j, y_j)\}_{j=1}^{|\mathcal{T}_i|}$.
4) Use SBI to update posterior $p(\theta | \mathcal{D} \cup \mathcal{D}_{i}^{hall})$
5) At candidate $x$
    - Sample $\theta_{k} \sim p(\theta | \mathcal{D} \cup \mathcal{D}_{i}^{hall})$.
    - Simulate $y_k \sim p(y|x, \theta_k)$
    - Use to calc entropy -- e.g. estimate variance/covariance + Gaussian entropy (NB we do this for NNs too)
6) Repeat for $i=1,\ldots,N$ posterior samples 


## Re: NN-BAX framework, can we split in the same way?
Maybe if we define things in the right way

Model training => SBI
Model sample => want x->y, so sample $\theta$, run sim for given x?

## Okay, let's try to set up a toy example

Assume we have an unknown function $f(x, \theta) = \theta\cdot x^2 + \epsilon$, where $\epsilon$ is some Gaussian noise. We can only measure at an individual $x$ point at a time, and want to use BOED to figure out where to measure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

from sbi.inference import NPE
from sbi.utils import BoxUniform

from sbi.utils.user_input_checks import (
    process_prior,
)

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [ ]:
# This is the true unknown theta value
theta_true = 4
# This is the amount of instrument noise
noise_true = 1
# Make the design space discrete
x_vals = torch.linspace(-10, 10, 100)


def simulator(x, theta, noise=0):
    fn_val = (x - theta) ** 2
    # Add Gaussian noise to the output
    fn_val += torch.normal(torch.zeros_like(x), torch.ones_like(x)).to(fn_val) * noise
    # Return both x and the function value concatenated
    return torch.cat((x, fn_val), dim=-1).to(fn_val)


def make_measurement(x, theta, noise):
    return simulator(x, theta, noise)[:, 1]

In [ ]:
# Plot the simulator output for a specific theta value
plt.figure(figsize=(10, 6))
plt.scatter(
    x_vals,
    make_measurement(x_vals, theta_true, noise_true),
    label="True Theta",
    color="red",
)
# Plot the simulator output for different theta values
for try_theta in np.linspace(-5, 5, 5):
    plt.plot(
        x_vals,
        make_measurement(x_vals, try_theta, 0),
        label=f"Theta={try_theta}",
        alpha=0.5,
        color="blue",
    )
plt.legend()
plt.title("Simulator Output for Different Theta Values")
plt.xlabel("x")
plt.ylabel("Simulator Output")
plt.show()

In [ ]:
# Construct the prior over our unknown parameter theta.
prior_theta = BoxUniform(low=torch.tensor([-5]), high=torch.tensor([5]), device=device)
# Check prior, return PyTorch prior.
prior_theta, num_parameters_theta, prior_theta_returns_numpy = process_prior(
    prior_theta
)

In [ ]:
from sbi.neural_nets import posterior_nn
from sbi.neural_nets.embedding_nets import FCEmbedding, PermutationInvariantEmbedding

# embedding
latent_dim = 10
single_trial_net = FCEmbedding(
    input_dim=2,
    num_hiddens=40,
    num_layers=2,
    output_dim=latent_dim,
)

embedding_net = PermutationInvariantEmbedding(
    single_trial_net,
    trial_net_output_dim=latent_dim,
    # NOTE: post-embedding is not needed really.
    num_layers=1,
    num_hiddens=10,
    output_dim=10,
)

# we choose a simple MDN as the density estimator.
density_estimator = posterior_nn("mdn", embedding_net=embedding_net)

In [ ]:
# Build the inference object.
inference = NPE(prior=prior_theta, density_estimator=density_estimator, device=device)

In [ ]:
# Make a training dataset
num_simulations = 2000
# Sample some thetas
theta_samples = prior_theta.sample((num_simulations,))
# Sample some x-values
x_samples = 0
# Simulate some samples
samples = simulator(x_samples, theta_samples)

In [ ]:
# Pass the data to the inference object
inference = inference.append_simulations(theta_samples, samples)

# Continue original from Sean

In [ ]:
# we need to fix the maximum number of trials.
max_num_trials = 20

# construct training data set: we want to cover the full range of possible number of
# trials
num_training_samples = 1000
theta_x = prior_joint.sample((num_training_samples,))

# there are certainly smarter ways to construct the training data set, but we go with a
# for loop here for illustration purposes.
y = torch.ones(
    num_training_samples * max_num_trials, max_num_trials, 1, device=device
) * float("nan")

for i in range(num_training_samples):
    yi = simulator(theta_x[i].repeat(max_num_trials, 1))
    for j in range(max_num_trials):
        y[i * max_num_trials + j, : j + 1, :] = yi[: j + 1, :]

theta_x = theta_x.repeat_interleave(max_num_trials, dim=0)


theta = theta_x[:, :1]

x = torch.ones(
    num_training_samples * max_num_trials, max_num_trials, 1, device=device
) * float("nan")

for i in range(num_training_samples):
    xi = theta_x[i, 1:].repeat(max_num_trials, 1)
    for j in range(max_num_trials):
        x[i * max_num_trials + j, : j + 1, :] = xi[: j + 1, :]

In [ ]:
context = torch.cat([y, x], dim=-1)
inference = inference.append_simulations(theta, context, exclude_invalid_x=False)

In [ ]:
density_estimator = inference.train(training_batch_size=1000)

In [ ]:
posterior = inference.build_posterior(density_estimator)

In [ ]:
def context_from_yx_lists(y_obs, theta_x):
    assert len(y_obs) == len(theta_x)
    if len(y_obs) == 0:
        return None

    all_context = []
    for idx in range(len(y_obs)):
        all_context.append(torch.cat([y_obs[idx], theta_x[idx][:, 1:]], dim=1))

    all_context = torch.cat(all_context, dim=0)

    trial_context = torch.ones(
        1, max_num_trials, all_context.shape[-1], device=device
    ) * float("nan")
    trial_context[0, : len(all_context), :] = all_context

    return trial_context

In [ ]:
cand_x_vals = torch.linspace(-10, 10, 20).to(device)
num_eig_samples = 200

theta_x_measured = []
y_obs = []

all_eigs = []
# Num acquisition iters
for iters in range(1):
    # Aiming to compute
    # EIG(x) = H[p(theta | D)] - E_{y~p(y | x, D)} H[p(theta | D U {(x, y)})]

    # Create context with new data
    context = context_from_yx_lists(y_obs, theta_x_measured)

    eigs = []
    # For each value of interest
    for candidate_x in tqdm(cand_x_vals):
        # First sample theta_i ~ p(theta | D) (or p(theta), if no data acquired)
        if context is None:
            post_samples = prior_theta.sample(
                (num_eig_samples,),
            )
        else:
            post_samples = posterior.sample(
                (num_eig_samples,), x=context, show_progress_bars=False
            )

        # Evaluate log(p(theta_i | D))
        if context is None:
            log_prob_nohall = prior_theta.log_prob(post_samples.unsqueeze(0))
        else:
            log_prob_nohall = posterior.log_prob_batched(
                post_samples.unsqueeze(0), x=context.repeat((len(post_samples), 1, 1))
            )

        # H[p(theta | D)] is approx -1/N * sum(log(p(theta_i | D)))
        entropy_no_hall = -log_prob_nohall.mean()

        # Use samples above of theta_i ~ p(theta | D)

        # Simulate y_i ~ p(y |x, D) for a given x. Conditioning on D comes via p(theta | D)
        sim_in = torch.cat(
            [post_samples, candidate_x[None, None].repeat((len(post_samples), 1))],
            dim=1,
        )

        y_i = sim_fn(sim_in)

        # Now hallucinate to get p(theta | D U {(x, y)})
        entropies_hall = []
        for idx in range(len(sim_in)):
            # Create new context D U {(x, y)} for each y_i
            theta_x_hall = theta_x_measured + [sim_in[idx : idx + 1]]
            y_obs_hall = y_obs + [y_i[idx : idx + 1]]
            context_hall = context_from_yx_lists(y_obs_hall, theta_x_hall)

            # Sample theta_j ~ p(theta | D U {(x, y)})
            samples_hall = posterior.sample(
                (num_eig_samples,), context_hall, show_progress_bars=False
            )

            # Compute entropy from those samples
            log_prob_hall = posterior.log_prob(samples_hall, x=context_hall)

            entropies_hall.append(-log_prob_hall.mean())

        # Take the expectation over p(theta | D U {(x, y)})
        mean_entropies_hall = torch.stack(entropies_hall).mean()

        eigs.append(entropy_no_hall - mean_entropies_hall)

    next_x_idx = torch.argmax(torch.stack(eigs))
    next_x = cand_x_vals[next_x_idx]
    next_theta_x_measured = torch.tensor([theta_true, next_x])[None].to(device)
    next_y_obs = simulator(next_theta_x_measured, noise=True)

    theta_x_measured += [next_theta_x_measured]
    y_obs += [next_y_obs]

    all_eigs.append(eigs)

In [ ]:
for n_trials in range(1, 20, 5):
    context = context_from_yx_lists(
        [y_obs[0]] * n_trials, [theta_x_measured[0]] * n_trials
    )

    samples = posterior.sample((10000,), context, show_progress_bars=False)

    plt.hist(samples.squeeze().cpu(), bins=20, range=[-5, 5], histtype="step")

plt.show()

In [ ]:
context.shape

In [ ]:
plt.scatter(context[0, :, 1].cpu(), context[0, :, 0].cpu())

for try_theta in [-4, 1, 4]:  # np.linspace(-5, 5, 10):
    theta_try = torch.ones(100) * try_theta
    theta_x_try = torch.cat([theta_try[:, None], x_vals[:, None]], dim=1)
    plt.plot(x_vals, simulator(theta_x_try), label=try_theta)
plt.legend()

In [ ]:
y_obs

In [ ]:
context = context_from_yx_lists(y_obs, theta_x_measured)

In [ ]:
plt.scatter(post_samples.squeeze().cpu(), log_prob_nohall.squeeze().cpu())

In [ ]:
for i in range(1, len(y_obs) + 1):
    context = context_from_yx_lists(y_obs[:i], theta_x_measured[:i])
    samples = posterior.sample((20000,), x=context)

    plt.hist(samples.squeeze().cpu(), bins=10, histtype="step")

plt.show()

In [ ]:
context = context_from_yx_lists(y_obs, theta_x_measured)
samples = posterior.sample((20000,), x=context)

plt.hist(samples.squeeze().cpu(), bins=10, histtype="step")

plt.show()

In [ ]:
true_theta_vals = torch.ones(cand_x_vals.shape[0]) * theta_true
theta_x_true = torch.cat([true_theta_vals[:, None], cand_x_vals.cpu()[:, None]], dim=1)

In [ ]:
plt.scatter(
    torch.stack(theta_x_measured).squeeze()[:, 1].cpu(),
    torch.stack(y_obs).flatten().cpu(),
)

plt.plot(theta_x_true.cpu()[:, 1], sim_fn(theta_x_true).cpu().flatten())

In [ ]:
plt.plot(cand_x_vals.cpu(), torch.stack(all_eigs[0]).cpu())
plt.plot(cand_x_vals.cpu(), torch.stack(all_eigs[-1]).cpu())

In [ ]:
for i in range(len(all_eigs)):
    plt.plot(cand_x_vals.cpu(), torch.stack(all_eigs[i]).cpu())

# I think not needed but still right?

\begin{equation}
EIG(x) = \mathbb{H}[\mathcal{T} |\mathcal{D}] - \mathbb{E}_{y\sim p(y|x, \mathcal{D})} \mathbb{H}[\mathcal{T} | \mathcal{D}, x, y]
\end{equation}

The first term is not a function of x, so may be dropped.

To compute the second term, for a given $x$, we 
1) Use SBI to estimate the posterior $p(\theta | \mathcal{D})$ (or start from prior $p(\theta)$ on iter 0).
2) Sample $\theta_i$ from $p(\theta | \mathcal{D})$ 
3) Simulate $y_i \sim p(y | \theta_i, x)$
4) Use SBI to estimate the posterior $p(\theta | \mathcal{D} \cup \{(x, y_i\})$.
5) Draw samples $\theta_{i,j} \sim p(\theta | \mathcal{D} \cup \{(x, y_i\})$
6) Use the simulator given $\theta_{i,j}$ to run the BAX algorithm.
    - E.g. argmax of some "quality function" $Q(y)$ on a grid of experimental settings $x_{j}$.
        - Evaluate simulator on all $x_j$, assuming $\theta_{i}$ (or $y_j \sim p(y | \theta_i, x_j)$).
        - Find $\arg\max_{x_j} Q(y_j)$ by explicitly comparing $y_j$
        
7) Rerun inner loop $j=1, \ldots, M$ times to estimate $p(\mathcal{T}| \mathcal{D}, x, y_i)$ and compute entropy.
8) Rerun outer loop $i=1, \ldots, N$ times to compute the expectation


Note -- this needs to be done for each eval point, so potentially pretty expensive.


## Distinction from GP/NN Case


Big questions are: why is formulation generally different from GP/NN, and why do we need to do this NxM thing at each point. In the functional case, what we do for term 2 is run the algorithm on a posterior sample N times and retrain on the result, namely

1) Sample $f_i \sim p(f | \mathcal{D})$
2) Run algo on $f \implies \mathcal{T}_i \sim p(\mathcal{T} | \mathcal{D})$
3) Retrain on ${T}_i \implies p(f | \mathcal{D}\cup {T}_i)$
4) Calc posterior predictive
5) For a given $x$, calc entropy